## Training the Open-Source LLM

In [1]:
!pip install -q --upgrade bitsandbytes trl

!wget https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 61.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 59.7 MB/s eta 0:00:00
--2026-10-20 13:50:50--  https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6917 (6.8K) [text/plain]
Saving to: ‘util.py’

util.py             100%[===================>]   6.75K  --.-KB/s    in 0s      

2026-10-20 13:50:50 (120 MB/s) - ‘util.py’ saved [6917/6917]



In [2]:
# libraries
import os
import re
import math
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, set_seed
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
import matplotlib.pyplot as plt
import wandb

In [3]:
# setting constants and environments
BASE_MODEL = 'meta-llama/Llama-3.2-3B'
PROJECT_NAME = 'price'
HF_USER = 'irene12'

LITE_MODE = False

DATA_USER = 'irene12'
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"

if LITE_MODE:
    RUN_NAME += "-lite"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# hyper-parameters
EPOCHS = 1 if LITE_MODE else 3
BATCH_SIZE = 32 if LITE_MODE else 256
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1

# QLoRA hyper-parameter
QUANT_4_BIT = True
LORA_R = 32 if LITE_MODE else 256
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ['q_proj', 'v_proj', 'k_proj', 'o_proj']
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT = 0.1

# training
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

capability = torch.cuda.get_device_capability()
user_bf16 = capability[0] >= 8

# tracking
VAL_SIZE = 500 if LITE_MODE else 1000
LOG_STEPS = 5 if LITE_MODE else 10
SAVE_STEPS = 100 if LITE_MODE else 200
LOG_TO_WANDB = True

In [ ]:
hf_token = 'HF_TOKEN'

login(hf_token)

In [ ]:
# logging into Weights & Biases
wandb_api_key = 'WANDB_API_KEY'
os.environ['WANDB_API_KEY'] = wandb_api_key
wandb.login()

# configuring Weights & Biases to record against our project
os.environ['WANDB_PROJECT'] = PROJECT_NAME
os.environ['WANDB_LOG_MODEL'] = 'false'
os.environ['WANDB_WATCH'] = 'false'

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: i-busah (i-busah-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
# loading the dataset
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
val = dataset['val'].select(range(VAL_SIZE))
test = dataset['test']

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  172MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/val-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.15MB            

data/val-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.18MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/800000 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [8]:
# len(train)

In [7]:
if LOG_TO_WANDB:
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)

### Loading the Tokenizer and Model

Loading the quantized model - reduced to 4 bits precision

In [ ]:
# user_bf16

False

In [10]:
# setting the quantization

if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if user_bf16 else torch.float16,
        bnb_4bit_quant_type='nf4'
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if user_bf16 else torch.float16
    )

In [11]:
# loading the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map='auto',
    dtype=torch.float16,
)

base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Memory footprint: 2197.6 MB


### Configuration for Training

- A LoRAConfig object with the hyper-parameters
- An SFTConfig with the overall training parameters

In [12]:
# LoRA Parameters
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=TARGET_MODULES
)

In [13]:
# training parameters

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=not user_bf16,
    bf16=user_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_steps=WARMUP_RATIO,
    train_sampling_strategy="group_by_length",
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy='steps',
    hub_strategy='every_save',
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy='steps',
    eval_steps=SAVE_STEPS
)

### Create the Trainer

In [14]:
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters
)

Adding EOS to train dataset:   0%|          | 0/800000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/800000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/800000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/800000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/800000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

### Fine-tuning the Base Model Now

In [ ]:
# fine-tuning the model
fine_tuning.train()

# pushing the fine-tuned model to hugging face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

In [17]:
if LOG_TO_WANDB:
    wandb.finish()

eval/entropy,▄▃█▇▁▂▂
eval/loss,█▄▅▂▁▁▁
eval/mean_token_accuracy,▁▅██▆▇▇
eval/num_tokens,▁▂▄▅▆██
eval/runtime,▁▇▆▄█▆█
eval/samples_per_second,█▂▃▅▁▃▁
eval/steps_per_second,█▂▃▅▁▃▁
train/entropy,█▂▂▂▁▁▁▁▁▁▁▁▂▂▁▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁
train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇███
train/global_step,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇████
+5,...
